# WorldBench Saved Video Demo

Use this notebook when you have two saved videos:

- a ground-truth future video
- a predicted or generated future video

WorldBench evaluates saved visual futures. It does not run model inference, robot control, or real-world task execution.

## 1. Install WorldBench

The normal path installs the latest PyPI package with video decoding support. If you need the current repository version instead, uncomment the fallback line.

In [ ]:
%pip install -q "worldbench[video]"

# Fallback for repository installs:
# %pip install -q "worldbench[video] @ git+https://github.com/tigee1311/worldbench.git"

## 2. Upload Two MP4 Files

Upload the ground-truth video first and the predicted-future video second. MP4 is the most portable format in Colab.

In [ ]:
from google.colab import files

uploaded = files.upload()
video_paths = list(uploaded.keys())
if len(video_paths) != 2:
    raise ValueError(f"Upload exactly two videos; got {len(video_paths)} file(s).")

ground_truth_video = video_paths[0]
prediction_video = video_paths[1]
print("Ground truth:", ground_truth_video)
print("Prediction:", prediction_video)

## 3. Run the Public CLI

This is the same command shown in the README. It writes JSON, Markdown, and a small visual comparison artifact.

In [ ]:
import subprocess
from pathlib import Path

output_dir = Path("worldbench_results")
cmd = [
    "worldbench",
    "eval-videos",
    "--ground-truth",
    ground_truth_video,
    "--prediction",
    prediction_video,
    "--output",
    str(output_dir),
]
completed = subprocess.run(cmd, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("WorldBench evaluation failed")

## 4. Inspect the Score Summary

The Composite Score summarizes available metrics for this aligned pair. It is not accuracy and not robot task success. Metrics that need actions, state, tracking, or contact metadata are reported as N/A.

In [ ]:
import json

result = json.loads((output_dir / "result.json").read_text())
print(f"Composite Score: {result['score']:.2f}/100")
print("Metric coverage:", result["coverage"]["available_metric_count"], "of", result["coverage"]["configured_metric_count"])
print()
for name, metric in result["metrics"].items():
    score = "N/A" if metric["score"] is None else f"{metric['score']:.1f}"
    reason = f" - {metric['reason']}" if metric.get("reason") else ""
    print(f"{name}: {score}{reason}")

## 5. Display the Visual Comparison

The contact sheet shows sampled ground-truth frames beside aligned prediction frames.

In [ ]:
from IPython.display import Image, display

comparison = output_dir / "artifacts" / "comparison.png"
if comparison.exists():
    display(Image(filename=str(comparison)))
else:
    print("No comparison image was generated.")